In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, confusion_matrix, precision_score, f1_score, accuracy_score, classification_report, roc_auc_score

In [3]:
# set experiment
mlflow.set_experiment("customer_churn_prediction")
mlflow.sklearn.autolog()

In [18]:
# import data
df = pd.read_csv('../data/Telco_Customer_Churn_cleaned.csv')


In [19]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [20]:
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [21]:
# train-test split

X = df.drop(columns = ['Churn'])
y = df['Churn']

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [23]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
num_cols = ["tenure", 'MonthlyCharges', 'TotalCharges']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)


In [24]:
# 1. Logistic Regression

In [26]:

with mlflow.start_run(run_name="LogisticRegression"):

    model = Pipeline(steps=[
        ('preprocessing', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, C=1))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_pred))

    mlflow.log_metric("test_f1", f1_score(y_test, y_pred))

    mlflow.log_metric("test_recall", recall_score(y_test, y_pred))

    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_prob))

2026/05/12 12:55:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
# Naive Bayes

In [27]:
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline

experiments = [
    {"var_smoothing": 1e-9},
    {"var_smoothing": 1e-8},
    {"var_smoothing": 1e-7},
    {"var_smoothing": 1e-6}
]

for exp in experiments:

    with mlflow.start_run(
        run_name=f"NB_{exp['var_smoothing']}"
    ):

        model = Pipeline(steps=[
            ('preprocessing', preprocessor),

            ('classifier',
             GaussianNB(**exp))
        ])

        # Train
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)

        # Probabilities
        y_prob = model.predict_proba(X_test)[:, 1]

        # Metrics
        mlflow.log_metric(
            "test_accuracy",
            accuracy_score(y_test, y_pred)
        )

        mlflow.log_metric(
            "test_f1",
            f1_score(y_test, y_pred)
        )

        mlflow.log_metric(
            "test_recall",
            recall_score(y_test, y_pred)
        )

        mlflow.log_metric(
            "test_roc_auc",
            roc_auc_score(y_test, y_prob)
        )

2026/05/12 12:56:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/12 12:56:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/12 12:56:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

In [ ]:
# 3. Random Forest

In [28]:
mlflow.sklearn.autolog()

experiments = [
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 200, "max_depth": 10}
]

for exp in experiments:

    with mlflow.start_run(
        run_name=f"RF_{exp['n_estimators']}_{exp['max_depth']}"
    ):

        model = Pipeline(steps=[
            ('preprocessing', preprocessor),
            ('classifier', RandomForestClassifier(**exp))
        ])

        # Train
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)

        # Probabilities
        y_prob = model.predict_proba(X_test)[:,1]

        # Metrics
        mlflow.log_metric(
            "test_accuracy",
            accuracy_score(y_test, y_pred)
        )

        mlflow.log_metric(
            "test_f1",
            f1_score(y_test, y_pred)
        )

        mlflow.log_metric(
            "test_recall",
            recall_score(y_test, y_pred)
        )

        mlflow.log_metric(
            "test_roc_auc",
            roc_auc_score(y_test, y_prob)
        )

2026/05/12 12:57:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/12 12:57:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
# 4. XGBoost

In [29]:
mlflow.sklearn.autolog()

# Different experiment combinations
experiments = [
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.05},
    {"n_estimators": 300, "max_depth": 7, "learning_rate": 0.01},
    {"n_estimators": 150, "max_depth": 4, "learning_rate": 0.2}
]

for exp in experiments:

    with mlflow.start_run(
        run_name=f"XGB_{exp['n_estimators']}_{exp['max_depth']}_{exp['learning_rate']}"
    ):

        model = Pipeline(steps=[
            ('preprocessing', preprocessor),

            ('classifier',
             XGBClassifier(
                 **exp,
                 random_state=42,
                 eval_metric='logloss'
             ))
        ])

        # Train
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)

        # Probabilities
        y_prob = model.predict_proba(X_test)[:, 1]

        # Metrics
        mlflow.log_metric(
            "test_accuracy",
            accuracy_score(y_test, y_pred)
        )

        mlflow.log_metric(
            "test_f1",
            f1_score(y_test, y_pred)
        )

        mlflow.log_metric(
            "test_recall",
            recall_score(y_test, y_pred)
        )

        mlflow.log_metric(
            "test_roc_auc",
            roc_auc_score(y_test, y_prob)
        )

2026/05/12 12:58:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/12 12:58:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/12 12:58:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

In [ ]:
# Model Selection

In [ ]:
# 

c:\Durva\Workspace\Projects\churn_project\notebooks
